# Sudoku Solver Project (Mathematics + Python)

**Final Project – Mathematics & Python**

**Author:** *Hristina Todorova*  
**Course:** *Math Concepts for Developers*  
**Date:** 30 March 2026

## Abstract

In this project, I explore Sudoku as a mathematical and algorithmic problem. The notebook models Sudoku as a **constraint satisfaction problem (CSP)** and investigates how different solving strategies behave in practice.

Three components are implemented in Python:
- a validity-checking framework,
- a naive recursive **backtracking** solver,
- an improved solver that uses the **minimum remaining values (MRV)** heuristic and simple **constraint propagation**.

The project combines maths, designing algorithms, and testing them out. The experiments compare the methods on puzzles of different difficulty and evaluate them based on how long they take, the number of recursive calls, and the number of assignments.

The results show that while naive backtracking is enough for simple puzzles, heuristics and constraint propagation can make the search much easier on harder instances. The project shows how maths can be used to turn a fun puzzle into a proper study of algorithms.

## 1. Introduction

I chose Sudoku because I used to solve it in my free time and always wondered whether there was a proper, systematic way to crack even the hardest puzzles — not just guessing and hoping for the best. When I encountered constraint satisfaction problems in this course, I realised that what I had been doing intuitively actually had a precise mathematical structure behind it. That connection motivated me to build a solver from scratch and measure how much smarter algorithms can help.

A standard Sudoku puzzle consists of a $9 \times 9$ grid partially filled with digits from 1 to 9. The goal is to complete the grid so that:
- each row contains every digit from 1 to 9 exactly once,
- each column contains every digit from 1 to 9 exactly once,
- each $3 \times 3$ subgrid contains every digit from 1 to 9 exactly once.

Although Sudoku is often viewed as entertainment, it can also be studied mathematically and algorithmically. It is a natural example of a **constraint satisfaction problem**, where variables must be assigned values that satisfy a set of restrictions.

This project treats Sudoku as a bridge between:
- discrete mathematics,
- combinatorics,
- algorithm design,
- computational experimentation.

The goal is not only to solve Sudoku, but to understand **why some methods are better than others**, and how mathematical structure helps reduce computational complexity.

## 2. Research Question

The main research question of this project is:

**How can Sudoku be modeled as a mathematical constraint satisfaction problem, and how much can heuristics and constraint propagation improve solving performance compared to naive backtracking?**

This question is relevant because Sudoku is simple to state but potentially expensive to solve by brute force. The project therefore investigates not just correctness, but also efficiency and algorithmic design.

## 3. Mathematical Formulation of Sudoku

### 3.1 Variables and domains

Let the Sudoku grid be represented by variables

$$
X_{r,c}, \quad r,c \in \{1,\dots,9\},
$$

where each variable corresponds to one cell in the grid.

Each variable takes values from the domain

$$
D_{r,c} \subseteq \{1,2,3,4,5,6,7,8,9\}.
$$

If a cell is already filled in the initial puzzle, then its domain contains exactly one value. Otherwise, its domain initially contains all values from 1 to 9 that are not ruled out by the constraints.

### 3.2 Constraints

A valid Sudoku solution must satisfy three families of constraints:

1. **Row constraints**  
   For each row, the digits 1–9 must appear exactly once.

2. **Column constraints**  
   For each column, the digits 1–9 must appear exactly once.

3. **Subgrid constraints**  
   For each $3 \times 3$ subgrid, the digits 1–9 must appear exactly once.

This makes Sudoku a **finite constraint satisfaction problem**.

### 3.3 Search space

A brute-force approach wouldn't work because there are too many possible completions. The number of empty cells makes the number of possible completions grow exponentially. The point of an efficient solver is to quickly narrow down the search options.

### 3.4 Constraint propagation intuition

Constraint propagation reduces the number of unassigned cells by getting rid of values that don't follow the existing assignments. In practice, this means finding the best possible values for empty cells and focusing the search on the parts of the board that are most difficult.

In [1]:
import time
from copy import deepcopy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (8, 5)

## 4. Mathematical Insight

We can think of Sudoku as a puzzle where we have to meet certain requirements. Each piece of information in the puzzle can only have a certain number of possible values, and these values must fit within a set of rules.
The more empty cells there are, the harder Sudoku becomes. This makes it hard to solve using a basic search method.
Heuristics (e.g. Minimum Remaining Values (MRV)) reduce the number of branches in the search tree, while constraint propagation reduces the number of possible values for the variables. This shows how maths can make algorithms much better.

## 5. Data Representation and Utility Functions

The Sudoku board (grid) is represented as a list of lists of integers.  
The value `0` denotes an empty cell.

In [2]:
EASY_PUZZLE = [
    [5, 3, 0, 0, 7, 0, 0, 0, 0],
    [6, 0, 0, 1, 9, 5, 0, 0, 0],
    [0, 9, 8, 0, 0, 0, 0, 6, 0],
    [8, 0, 0, 0, 6, 0, 0, 0, 3],
    [4, 0, 0, 8, 0, 3, 0, 0, 1],
    [7, 0, 0, 0, 2, 0, 0, 0, 6],
    [0, 6, 0, 0, 0, 0, 2, 8, 0],
    [0, 0, 0, 4, 1, 9, 0, 0, 5],
    [0, 0, 0, 0, 8, 0, 0, 7, 9],
]

MEDIUM_PUZZLE = [
    [0, 2, 0, 6, 0, 8, 0, 0, 0],
    [5, 8, 0, 0, 0, 9, 7, 0, 0],
    [0, 0, 0, 0, 4, 0, 0, 0, 0],
    [3, 7, 0, 0, 0, 0, 5, 0, 0],
    [6, 0, 0, 0, 0, 0, 0, 0, 4],
    [0, 0, 8, 0, 0, 0, 0, 1, 3],
    [0, 0, 0, 0, 2, 0, 0, 0, 0],
    [0, 0, 9, 8, 0, 0, 0, 3, 6],
    [0, 0, 0, 3, 0, 6, 0, 9, 0],
]

HARD_PUZZLE = [
    [0, 0, 0, 0, 0, 0, 0, 1, 2],
    [0, 0, 0, 0, 3, 5, 0, 0, 0],
    [0, 0, 1, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 5, 0, 0, 4, 0, 7],
    [0, 0, 4, 0, 0, 0, 2, 0, 0],
    [9, 0, 7, 0, 0, 2, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 8, 0, 0],
    [0, 0, 0, 7, 6, 0, 0, 0, 0],
    [6, 3, 0, 0, 0, 0, 0, 0, 0],
]

PUZZLES = {"Easy": EASY_PUZZLE, "Medium": MEDIUM_PUZZLE, "Hard": HARD_PUZZLE}

In [ ]:
# print the Sudoku grid in a readable format
def print_grid(grid):
    for i, row in enumerate(grid):
        # add a visual separator every 3 rows to show the 3x3 blocks
        if i % 3 == 0 and i != 0:
            print("-" * 21)
        line = []
        for j, val in enumerate(row):
            # add a vertical separator every 3 columns
            if j % 3 == 0 and j != 0:
                line.append("|")
            # empty cells show as '.' so the grid is easier to read
            line.append(str(val) if val != 0 else ".")
        print(" ".join(line))

# convert board to pandas DataFrame for nicer display
def grid_dataframe(grid):
    df = pd.DataFrame(grid, columns=[f"C{j+1}" for j in range(9)])
    df.index = [f"R{i+1}" for i in range(9)]
    return df

# check if placing 'num' at (row, col) would break any constraint
def is_valid(grid, row, col, num):

    # the same number can't appear twice in the same row
    if any(grid[row][j] == num for j in range(9)):
        return False
    
    # the same number can't appear twice in the same column
    if any(grid[i][col] == num for i in range(9)):
        return False
    
    # the same number can't appear twice in the 3x3 box
    box_row = (row // 3) * 3
    box_col = (col // 3) * 3

    for i in range(box_row, box_row + 3):
        for j in range(box_col, box_col + 3):
            if grid[i][j] == num:
                return False
    return True

# scan the grid top-left to bottom-right and return the first empty cell (value = 0)
def find_empty_cell(grid):
    for i in range(9):
        for j in range(9):
            if grid[i][j] == 0:
                return i, j
    return None  # no empty cells → puzzle is fully filled

# the puzzle is complete when there are no empty cells left
def is_complete(grid):
    return find_empty_cell(grid) is None

In [4]:
print("Example: Easy puzzle")
print_grid(EASY_PUZZLE)

Example: Easy puzzle
5 3 . | . 7 . | . . .
6 . . | 1 9 5 | . . .
. 9 8 | . . . | . 6 .
---------------------
8 . . | . 6 . | . . 3
4 . . | 8 . 3 | . . 1
7 . . | . 2 . | . . 6
---------------------
. 6 . | . . . | 2 8 .
. . . | 4 1 9 | . . 5
. . . | . 8 . | . 7 9


## 6. Method 1: Naive Backtracking

Backtracking is the classical recursive method for solving Sudoku.

The idea is:
1. Find the next empty cell.
2. Try digits 1 through 9.
3. Keep a digit if it is valid.
4. Recurse.
5. If a contradiction is reached, undo the assignment and try another value.

This guarantees correctness, but can be expensive because it may explore a large search tree.

In [5]:
class SolverStats:
    def __init__(self):
        self.recursive_calls = 0
        self.assignments = 0
        self.backtracks = 0
        self.propagations = 0

# recursive backtracking solver
def solve_backtracking(grid, stats=None):
    if stats is None:
        stats = SolverStats()

    # count recursive calls
    stats.recursive_calls += 1

    # find next empty cell
    empty = find_empty_cell(grid)

    # if no empty cell → solution found
    if empty is None:
        return True, grid, stats

    row, col = empty

    # try all numbers from 1 to 9
    for num in range(1, 10):

        # check if number is valid
        if is_valid(grid, row, col, num):
            
            # assign number
            grid[row][col] = num
            stats.assignments += 1

            # recursive call
            solved, solved_grid, stats = solve_backtracking(grid, stats)

            if solved:
                return True, solved_grid, stats

            # undo assignment (backtrack)
            grid[row][col] = 0
            stats.backtracks += 1

    # no valid number found → backtrack
    return False, grid, stats

## 7. Method 2: Candidates and Constraint Propagation

Instead of guessing right away, a solver can first calculate the possible answers for each empty cell.  
If a cell has only one candidate, the value is forced and can be assigned immediately.

This is a simple but useful form of **constraint propagation**.

In [6]:
# get all possible values for a cell
def get_candidates(grid, row, col):
    
    # if already filled → no candidates
    if grid[row][col] != 0:
        return set()
    
    # start with all numbers
    candidates = set(range(1, 10))
    
    # remove numbers already in row
    candidates -= set(grid[row][j] for j in range(9))

    # remove numbers already in column
    candidates -= set(grid[i][col] for i in range(9))

    # remove numbers already in 3x3 box
    box_row = (row // 3) * 3
    box_col = (col // 3) * 3
    
    for i in range(box_row, box_row + 3):
        for j in range(box_col, box_col + 3):
            candidates.discard(grid[i][j])

    return candidates

# apply constraint propagation (fill forced cells)
def propagate_constraints(grid, stats=None):
    changed = True
    while changed:
        changed = False
        for i in range(9):
            for j in range(9):
                if grid[i][j] == 0:
                    candidates = get_candidates(grid, i, j)

                    # contradiction → no solution
                    if len(candidates) == 0:
                        return False, grid
                    
                    # only one possible value → fill it
                    if len(candidates) == 1:
                        grid[i][j] = next(iter(candidates))
                        changed = True

                        if stats is not None:
                            stats.assignments += 1
                            stats.propagations += 1
                            
    return True, grid